# Phase 4: Portfolio Construction & Risk Management  
## Notebook 04_12 — Black-Litterman Allocation

### Research question

How can we stabilise expected returns for portfolio optimisation by blending market-implied equilibrium returns with explicit investor views?

This notebook follows:

```text
04_04_mean_variance_optimization_ledoit_wolf
04_09_risk_parity_and_equal_risk_contribution
04_10_hierarchical_risk_parity
04_11_cvar_convex_optimization
```

Mean-variance optimisation is powerful but extremely sensitive to expected returns. The Black-Litterman model addresses this by starting from equilibrium returns implied by the market portfolio, then blending in investor views with uncertainty.

It covers:

1. market-cap-weighted equilibrium portfolio;
2. risk aversion estimation;
3. implied equilibrium excess returns;
4. absolute views;
5. relative views;
6. view matrix $P$;
7. view vector $Q$;
8. view uncertainty matrix $\Omega$;
9. posterior expected returns;
10. posterior covariance;
11. mean-variance optimisation using Black-Litterman returns;
12. comparison with sample-mean MVO, equal weight, risk parity-style baselines;
13. sensitivity to $\tau$;
14. sensitivity to view confidence;
15. active weights versus market portfolio;
16. out-of-sample validation;
17. rolling Black-Litterman allocation;
18. view attribution and governance checks;
19. saved outputs and manifest.

The key idea:

> Black-Litterman makes expected returns less arbitrary by anchoring them to market equilibrium and then carefully tilting them with quantified views.

## 1. Why Black-Litterman?

Classic mean-variance optimisation needs:

$$
\mu
$$

expected returns.

But expected returns are extremely noisy.

Small changes in $\mu$ can produce huge changes in optimal weights.

Black-Litterman starts with market-implied equilibrium excess returns:

$$
\pi = \lambda \Sigma w_m
$$

where:

- $\lambda$ is risk aversion;
- $\Sigma$ is covariance;
- $w_m$ is market portfolio weight.

Then it blends these priors with investor views.

Instead of asking:

> What exact return do I forecast for every asset?

Black-Litterman asks:

> What does the market imply, and what views do I have with confidence?

## 2. Equilibrium returns

If the market portfolio is mean-variance efficient, then the implied excess return vector is:

$$
\pi = \lambda\Sigma w_m
$$

This is reverse optimisation.

Rather than solving for weights from returns, it solves for returns from weights.

The market portfolio becomes the neutral starting point.

## 3. Investor views

Investor views are encoded as:

$$
P\mu = Q
$$

where:

- $P$ maps assets to views;
- $Q$ is the expected return value of each view.

Examples:

### Absolute view

```text
US equities will return 6% annualised excess.
```

$$
P = [1,0,0,\dots]
$$

### Relative view

```text
US equities will outperform European equities by 2%.
```

$$
P = [1,-1,0,\dots]
$$

View uncertainty is:

$$
\Omega
$$

A confident view has lower variance in $\Omega$.

An uncertain view has higher variance.

## 4. Black-Litterman posterior

Prior:

$$
\mu \sim N(\pi,\tau\Sigma)
$$

Views:

$$
Q = P\mu + \epsilon
$$

$$
\epsilon \sim N(0,\Omega)
$$

Posterior expected returns:

$$
\begin{aligned}
\mu_{BL} &= \Big[ (\tau\Sigma)^{-1} \\
&\quad + P^\top\Omega^{-1}P \Big]^{-1} \Big[ (\tau\Sigma)^{-1}\pi \\
&\quad + P^\top\Omega^{-1}Q \Big]
\end{aligned}
$$

Posterior covariance of expected returns:

$$
\begin{aligned}
M &= \Big[ (\tau\Sigma)^{-1} \\
&\quad + P^\top\Omega^{-1}P \Big]^{-1}
\end{aligned}
$$

A common adjusted return covariance for optimisation is:

$$
\Sigma_{BL}=\Sigma+M
$$

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.optimize import minimize
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

try:
    from sklearn.covariance import LedoitWolf
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

SCIPY_AVAILABLE, SKLEARN_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class BlackLittermanConfig:
    n_days: int = 2_400
    train_fraction: float = 0.65
    annualisation: int = 252
    tau: float = 0.05
    risk_free_rate_ann: float = 0.02
    risk_aversion_default: float = 3.0
    max_weight: float = 0.35
    min_weight: float = 0.00
    transaction_cost_bps: float = 3.0
    rebalance_window: int = 504
    rebalance_step: int = 21
    seed: int = 42


config = BlackLittermanConfig()
config

## 6. Synthetic global multi-asset universe

We simulate a global allocation universe:

- US equities;
- Europe equities;
- Japan equities;
- emerging equities;
- bonds;
- commodities;
- trend-following;
- real assets;
- crypto proxy.

We also simulate market capitalisation weights, which serve as the neutral market portfolio.

In [ ]:
def simulate_bl_universe(config: BlackLittermanConfig):
    rng = np.random.default_rng(config.seed)

    dates = pd.bdate_range("2014-01-01", periods=config.n_days)

    assets = [
        "US_EQ", "EU_EQ", "JP_EQ", "EM_EQ",
        "US_BOND", "EU_BOND",
        "GOLD", "OIL", "COPPER",
        "FX_CARRY", "TREND",
        "REIT", "BTC_PROXY"
    ]

    asset_class = {
        "US_EQ": "equity",
        "EU_EQ": "equity",
        "JP_EQ": "equity",
        "EM_EQ": "equity",
        "US_BOND": "bond",
        "EU_BOND": "bond",
        "GOLD": "commodity",
        "OIL": "commodity",
        "COPPER": "commodity",
        "FX_CARRY": "fx",
        "TREND": "alternative",
        "REIT": "real_asset",
        "BTC_PROXY": "crypto",
    }

    market_caps = pd.Series({
        "US_EQ": 45.0,
        "EU_EQ": 15.0,
        "JP_EQ": 7.0,
        "EM_EQ": 8.0,
        "US_BOND": 40.0,
        "EU_BOND": 20.0,
        "GOLD": 4.0,
        "OIL": 3.0,
        "COPPER": 2.0,
        "FX_CARRY": 3.0,
        "TREND": 4.0,
        "REIT": 5.0,
        "BTC_PROXY": 1.0,
    })

    market_weights = market_caps / market_caps.sum()

    factor_names = ["global_equity", "rates", "commodity", "carry", "trend", "crypto", "real_asset"]
    factor_vol_ann = np.array([0.14, 0.055, 0.13, 0.08, 0.09, 0.38, 0.12])
    factor_vol_daily = factor_vol_ann / np.sqrt(config.annualisation)

    factor_corr = np.array([
        [ 1.00, -0.25,  0.22,  0.15, -0.12,  0.35,  0.55],
        [-0.25,  1.00,  0.10, -0.05,  0.15, -0.20, -0.20],
        [ 0.22,  0.10,  1.00,  0.10,  0.05,  0.20,  0.20],
        [ 0.15, -0.05,  0.10,  1.00,  0.00,  0.15,  0.10],
        [-0.12,  0.15,  0.05,  0.00,  1.00,  0.00, -0.05],
        [ 0.35, -0.20,  0.20,  0.15,  0.00,  1.00,  0.30],
        [ 0.55, -0.20,  0.20,  0.10, -0.05,  0.30,  1.00],
    ])

    factor_cov = np.outer(factor_vol_daily, factor_vol_daily) * factor_corr

    loadings = pd.DataFrame(0.0, index=assets, columns=factor_names)
    loadings.loc[["US_EQ", "EU_EQ", "JP_EQ", "EM_EQ"], "global_equity"] = [1.00, 0.95, 0.80, 1.20]
    loadings.loc[["US_BOND", "EU_BOND"], "rates"] = [1.00, 0.90]
    loadings.loc[["GOLD", "OIL", "COPPER"], "commodity"] = [0.45, 1.10, 0.95]
    loadings.loc["FX_CARRY", "carry"] = 1.00
    loadings.loc["TREND", "trend"] = 1.00
    loadings.loc["TREND", "global_equity"] = -0.25
    loadings.loc["BTC_PROXY", "crypto"] = 1.00
    loadings.loc["BTC_PROXY", "global_equity"] = 0.40
    loadings.loc["REIT", "real_asset"] = 1.00
    loadings.loc["REIT", "global_equity"] = 0.60
    loadings.loc["REIT", "rates"] = -0.30
    loadings.loc["GOLD", "rates"] = 0.25

    idio_vol_ann = pd.Series({
        "US_EQ": 0.07,
        "EU_EQ": 0.08,
        "JP_EQ": 0.09,
        "EM_EQ": 0.13,
        "US_BOND": 0.025,
        "EU_BOND": 0.030,
        "GOLD": 0.12,
        "OIL": 0.22,
        "COPPER": 0.18,
        "FX_CARRY": 0.07,
        "TREND": 0.08,
        "REIT": 0.11,
        "BTC_PROXY": 0.52,
    })

    annual_mean = pd.Series({
        "US_EQ": 0.070,
        "EU_EQ": 0.060,
        "JP_EQ": 0.045,
        "EM_EQ": 0.080,
        "US_BOND": 0.025,
        "EU_BOND": 0.020,
        "GOLD": 0.035,
        "OIL": 0.045,
        "COPPER": 0.040,
        "FX_CARRY": 0.030,
        "TREND": 0.050,
        "REIT": 0.060,
        "BTC_PROXY": 0.120,
    })

    regime = np.zeros(config.n_days, dtype=int)
    transition = np.array([[0.985, 0.015], [0.055, 0.945]])

    asset_returns = np.zeros((config.n_days, len(assets)))
    factor_returns = np.zeros((config.n_days, len(factor_names)))
    stress_type = np.array(["normal"] * config.n_days, dtype=object)

    for t in range(config.n_days):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_multiplier = 1.0 if regime[t] == 0 else 2.3
        cov_t = factor_cov * vol_multiplier**2

        z = rng.standard_t(df=6, size=len(factor_names)) * np.sqrt((6 - 2) / 6)
        L = np.linalg.cholesky(cov_t + 1e-12 * np.eye(len(factor_names)))
        f = L @ z

        u = rng.uniform()
        if u < 0.009:
            stress_type[t] = "global_equity_crash"
            f[0] -= rng.uniform(0.035, 0.100)
            f[1] += rng.uniform(0.004, 0.025)
            f[5] -= rng.uniform(0.080, 0.220)
            f[4] += rng.uniform(0.005, 0.040)
        elif u < 0.015:
            stress_type[t] = "inflation_shock"
            f[1] -= rng.uniform(0.010, 0.040)
            f[2] += rng.uniform(0.030, 0.100)
            f[0] -= rng.uniform(0.010, 0.050)
        elif u < 0.021:
            stress_type[t] = "commodity_crash"
            f[2] -= rng.uniform(0.040, 0.130)
            f[0] -= rng.uniform(0.005, 0.030)
        elif u < 0.026:
            stress_type[t] = "crypto_crash"
            f[5] -= rng.uniform(0.150, 0.350)
            f[0] -= rng.uniform(0.005, 0.030)

        eps = rng.standard_t(df=5, size=len(assets)) * np.sqrt((5 - 2) / 5)
        eps = eps * (idio_vol_ann.values / np.sqrt(config.annualisation))

        asset_returns[t] = annual_mean.values / config.annualisation + loadings.to_numpy() @ f + eps
        factor_returns[t] = f

    returns_df = pd.DataFrame(asset_returns, columns=assets)
    returns_df.insert(0, "date", dates)
    returns_df["regime"] = regime
    returns_df["stress_type"] = stress_type

    factor_df = pd.DataFrame(factor_returns, columns=factor_names)
    factor_df.insert(0, "date", dates)
    factor_df["regime"] = regime
    factor_df["stress_type"] = stress_type

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "market_cap_proxy": [market_caps[a] for a in assets],
        "market_weight": [market_weights[a] for a in assets],
        "annual_mean_assumption": [annual_mean[a] for a in assets],
        "idio_vol_ann": [idio_vol_ann[a] for a in assets],
    })

    return returns_df, factor_df, metadata


returns_df, factor_returns, asset_metadata = simulate_bl_universe(config)
asset_cols = asset_metadata["asset"].tolist()
returns = returns_df.set_index("date")[asset_cols]
market_weights = asset_metadata.set_index("asset")["market_weight"].reindex(asset_cols)

returns_df.head(), asset_metadata

In [ ]:
plt.figure(figsize=(12, 6))
for asset in asset_cols:
    plt.plot(returns.index, (1 + returns[asset]).cumprod(), label=asset, alpha=0.75)
plt.title("Synthetic Asset Growth of 1")
plt.xlabel("Date")
plt.ylabel("Growth")
plt.legend(ncol=3)
plt.show()

plt.figure(figsize=(10, 6))
plt.barh(market_weights.sort_values().index, market_weights.sort_values().values)
plt.title("Market-Cap Proxy Weights")
plt.xlabel("Weight")
plt.ylabel("Asset")
plt.show()

## 7. Train/test split and covariance estimation

We estimate covariance from the training sample.

We use Ledoit-Wolf shrinkage when available.

In [ ]:
split_idx = int(len(returns) * config.train_fraction)
train_returns = returns.iloc[:split_idx]
test_returns = returns.iloc[split_idx:]

def estimate_covariance(returns_window):
    X = returns_window.dropna().to_numpy()

    if SKLEARN_AVAILABLE:
        estimator = LedoitWolf().fit(X)
        cov = estimator.covariance_
        method = "sklearn_ledoit_wolf"
        shrinkage = float(estimator.shrinkage_)
    else:
        sample = np.cov(X, rowvar=False, ddof=1)
        target = np.diag(np.diag(sample))
        shrinkage = 0.30
        cov = shrinkage * target + (1 - shrinkage) * sample
        method = "fallback_diagonal_shrinkage"

    return pd.DataFrame(cov, index=returns_window.columns, columns=returns_window.columns), method, shrinkage


cov_train, cov_method, cov_shrinkage = estimate_covariance(train_returns)

split_metadata = {
    "n_total_days": int(len(returns)),
    "n_train_days": int(len(train_returns)),
    "n_test_days": int(len(test_returns)),
    "train_start": str(train_returns.index[0]),
    "train_end": str(train_returns.index[-1]),
    "test_start": str(test_returns.index[0]),
    "test_end": str(test_returns.index[-1]),
    "covariance_method": cov_method,
    "covariance_shrinkage": cov_shrinkage,
}

pd.Series(split_metadata)

## 8. Estimate market risk aversion

If market portfolio excess return is $R_m-R_f$, a simple risk aversion estimate is:

$$
\lambda = \frac{E[R_m-R_f]}{Var(R_m)}
$$

We also keep a default risk-aversion value because this estimate can be noisy.

In [ ]:
market_portfolio_return_train = train_returns @ market_weights
rf_daily = config.risk_free_rate_ann / config.annualisation
market_excess_train = market_portfolio_return_train - rf_daily

lambda_estimated = market_excess_train.mean() / market_excess_train.var(ddof=1)
lambda_used = config.risk_aversion_default if not np.isfinite(lambda_estimated) or lambda_estimated <= 0 else lambda_estimated

risk_aversion_report = pd.Series({
    "lambda_estimated_from_train": lambda_estimated,
    "lambda_used": lambda_used,
    "market_portfolio_mean_excess_ann": market_excess_train.mean() * config.annualisation,
    "market_portfolio_vol_ann": market_portfolio_return_train.std(ddof=1) * np.sqrt(config.annualisation),
})

risk_aversion_report

## 9. Implied equilibrium excess returns

Reverse optimisation:

$$
\pi = \lambda \Sigma w_m
$$

This produces annualised implied excess returns:

$$
252\pi
$$

In [ ]:
def implied_equilibrium_returns(cov_daily, market_weights, risk_aversion):
    pi_daily = risk_aversion * cov_daily.to_numpy() @ market_weights.to_numpy()
    return pd.Series(pi_daily, index=cov_daily.index)


pi_daily = implied_equilibrium_returns(cov_train, market_weights, lambda_used)
pi_ann = pi_daily * config.annualisation

pi_table = pd.DataFrame({
    "asset": asset_cols,
    "market_weight": market_weights.reindex(asset_cols).values,
    "pi_implied_excess_return_ann": pi_ann.reindex(asset_cols).values,
    "sample_excess_return_ann": (train_returns.mean() - rf_daily).reindex(asset_cols).values * config.annualisation,
})

pi_table.sort_values("pi_implied_excess_return_ann", ascending=False)

In [ ]:
plt.figure(figsize=(12, 6))
x = np.arange(len(asset_cols))
plt.bar(x - 0.2, pi_table["pi_implied_excess_return_ann"], width=0.4, label="implied equilibrium")
plt.bar(x + 0.2, pi_table["sample_excess_return_ann"], width=0.4, label="sample excess mean")
plt.axhline(0, linestyle="--")
plt.xticks(x, asset_cols, rotation=45, ha="right")
plt.title("Expected Return Inputs")
plt.ylabel("Annualised excess return")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Investor views

We define example views:

1. **Absolute view**  
   Trend-following has 4% annualised excess return.

2. **Relative equity view**  
   US equities outperform European equities by 1.5% annualised.

3. **Relative commodity view**  
   Gold outperforms oil by 2% annualised.

4. **Crypto caution view**  
   Crypto proxy has 3% lower excess return than implied equilibrium.

5. **Bond relative view**  
   US bonds outperform European bonds by 0.75% annualised.

These are synthetic research views, not investment advice.

In [ ]:
def build_views(asset_cols, pi_daily, config):
    views = []

    def row_for_assets(weights_dict):
        row = pd.Series(0.0, index=asset_cols)
        for asset, weight in weights_dict.items():
            row.loc[asset] = weight
        return row

    # 1. Absolute view: TREND excess return = 4% annualised.
    views.append({
        "view_name": "absolute_trend_4pct",
        "P": row_for_assets({"TREND": 1.0}),
        "Q_ann": 0.040,
        "confidence": 0.70,
        "description": "TREND has 4% annualised excess return."
    })

    # 2. US equity outperforms EU equity by 1.5%.
    views.append({
        "view_name": "us_eq_outperforms_eu_eq",
        "P": row_for_assets({"US_EQ": 1.0, "EU_EQ": -1.0}),
        "Q_ann": 0.015,
        "confidence": 0.60,
        "description": "US equities outperform European equities by 1.5% annualised."
    })

    # 3. Gold outperforms oil by 2%.
    views.append({
        "view_name": "gold_outperforms_oil",
        "P": row_for_assets({"GOLD": 1.0, "OIL": -1.0}),
        "Q_ann": 0.020,
        "confidence": 0.55,
        "description": "Gold outperforms oil by 2% annualised."
    })

    # 4. Crypto caution: BTC_PROXY excess return is pi - 3%.
    views.append({
        "view_name": "crypto_caution",
        "P": row_for_assets({"BTC_PROXY": 1.0}),
        "Q_ann": float(pi_daily["BTC_PROXY"] * config.annualisation - 0.030),
        "confidence": 0.50,
        "description": "Crypto proxy expected excess return is 3% below equilibrium-implied return."
    })

    # 5. US bonds outperform EU bonds by 0.75%.
    views.append({
        "view_name": "us_bond_outperforms_eu_bond",
        "P": row_for_assets({"US_BOND": 1.0, "EU_BOND": -1.0}),
        "Q_ann": 0.0075,
        "confidence": 0.65,
        "description": "US bonds outperform European bonds by 0.75% annualised."
    })

    P = pd.DataFrame([v["P"] for v in views], index=[v["view_name"] for v in views])
    Q_daily = pd.Series([v["Q_ann"] / config.annualisation for v in views], index=P.index)
    confidence = pd.Series([v["confidence"] for v in views], index=P.index)

    view_table = pd.DataFrame({
        "view_name": P.index,
        "Q_ann": Q_daily.values * config.annualisation,
        "confidence": confidence.values,
        "description": [v["description"] for v in views],
    })

    return P, Q_daily, confidence, view_table


P, Q_daily, view_confidence, view_table = build_views(asset_cols, pi_daily, config)

view_table

In [ ]:
P

## 11. View uncertainty matrix $\Omega$

A common approach is:

$$
\Omega = diag(P(\tau\Sigma)P^\top)
$$

Then scale uncertainty by confidence.

Higher confidence means lower uncertainty.

We use:

$$
\Omega_i = \frac{1-c_i}{c_i} \cdot P_i(\tau\Sigma)P_i^\top
$$

with a floor for numerical stability.

In [ ]:
def build_omega(P, cov_daily, tau, confidence):
    P_mat = P.to_numpy()
    base = P_mat @ (tau * cov_daily.to_numpy()) @ P_mat.T
    base_diag = np.diag(base).copy()

    conf = confidence.reindex(P.index).to_numpy()
    conf = np.clip(conf, 0.01, 0.99)

    scaled_diag = base_diag * (1 - conf) / conf
    scaled_diag = np.maximum(scaled_diag, 1e-10)

    omega = pd.DataFrame(
        np.diag(scaled_diag),
        index=P.index,
        columns=P.index
    )

    return omega


omega = build_omega(P, cov_train, config.tau, view_confidence)

omega

## 12. Black-Litterman posterior calculation

We implement:

$$
\begin{aligned}
\mu_{BL} &= \Big[ (\tau\Sigma)^{-1} \\
&\quad + P^\top\Omega^{-1}P \Big]^{-1} \Big[ (\tau\Sigma)^{-1}\pi \\
&\quad + P^\top\Omega^{-1}Q \Big]
\end{aligned}
$$

and:

$$
\Sigma_{BL} = \Sigma + M
$$

where:

$$
\begin{aligned}
M &= \Big[ (\tau\Sigma)^{-1} \\
&\quad + P^\top\Omega^{-1}P \Big]^{-1}
\end{aligned}
$$

In [ ]:
def black_litterman_posterior(pi_daily, cov_daily, P, Q_daily, omega, tau):
    Sigma = cov_daily.to_numpy()
    pi = pi_daily.reindex(cov_daily.index).to_numpy().reshape(-1, 1)
    P_mat = P.reindex(columns=cov_daily.index).to_numpy()
    Q = Q_daily.reindex(P.index).to_numpy().reshape(-1, 1)
    Omega = omega.reindex(index=P.index, columns=P.index).to_numpy()

    tau_Sigma = tau * Sigma

    inv_tau_sigma = np.linalg.pinv(tau_Sigma)
    inv_omega = np.linalg.pinv(Omega)

    middle = inv_tau_sigma + P_mat.T @ inv_omega @ P_mat
    M = np.linalg.pinv(middle)

    posterior_mu = M @ (inv_tau_sigma @ pi + P_mat.T @ inv_omega @ Q)
    posterior_cov = Sigma + M

    mu_series = pd.Series(posterior_mu.flatten(), index=cov_daily.index)
    posterior_cov_df = pd.DataFrame(posterior_cov, index=cov_daily.index, columns=cov_daily.columns)
    M_df = pd.DataFrame(M, index=cov_daily.index, columns=cov_daily.columns)

    return mu_series, posterior_cov_df, M_df


mu_bl_daily, cov_bl_daily, posterior_mean_cov = black_litterman_posterior(
    pi_daily,
    cov_train,
    P,
    Q_daily,
    omega,
    config.tau
)

bl_return_table = pd.DataFrame({
    "asset": asset_cols,
    "market_weight": market_weights.reindex(asset_cols).values,
    "pi_ann": pi_daily.reindex(asset_cols).values * config.annualisation,
    "bl_mu_ann": mu_bl_daily.reindex(asset_cols).values * config.annualisation,
    "sample_mu_ann": (train_returns.mean().reindex(asset_cols).values - rf_daily) * config.annualisation,
})

bl_return_table["bl_minus_pi_ann"] = bl_return_table["bl_mu_ann"] - bl_return_table["pi_ann"]

bl_return_table.sort_values("bl_minus_pi_ann", ascending=False)

In [ ]:
plt.figure(figsize=(12, 6))
x = np.arange(len(asset_cols))
plt.bar(x - 0.25, bl_return_table["pi_ann"], width=0.25, label="equilibrium pi")
plt.bar(x, bl_return_table["bl_mu_ann"], width=0.25, label="Black-Litterman")
plt.bar(x + 0.25, bl_return_table["sample_mu_ann"], width=0.25, label="sample mean")
plt.axhline(0, linestyle="--")
plt.xticks(x, asset_cols, rotation=45, ha="right")
plt.title("Expected Return Comparison")
plt.ylabel("Annualised excess return")
plt.legend()
plt.tight_layout()
plt.show()

## 13. Portfolio optimisation utilities

We solve:

$$
\max_w \quad w^\top\mu - \frac{\lambda}{2}w^\top\Sigma w
$$

subject to:

$$
\sum_i w_i=1
$$

$$
w_i\in[w_{min},w_{max}]
$$

This is a constrained mean-variance utility problem.

In [ ]:
def portfolio_return(weights, mu):
    return float(np.asarray(weights) @ np.asarray(mu))


def portfolio_volatility(weights, cov):
    w = np.asarray(weights, dtype=float)
    cov_arr = np.asarray(cov, dtype=float)
    return float(np.sqrt(max(w.T @ cov_arr @ w, 0.0)))


def optimise_mean_variance(mu_daily, cov_daily, risk_aversion, min_weight=0.0, max_weight=0.35):
    mu = np.asarray(mu_daily, dtype=float)
    cov = np.asarray(cov_daily, dtype=float)
    n = len(mu)
    x0 = np.ones(n) / n

    def obj(w):
        return -(w @ mu - 0.5 * risk_aversion * (w.T @ cov @ w))

    if SCIPY_AVAILABLE:
        res = minimize(
            obj,
            x0=x0,
            method="SLSQP",
            bounds=[(min_weight, max_weight)] * n,
            constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}],
            options={"maxiter": 1000, "ftol": 1e-12}
        )

        if res.success:
            w = res.x
            return pd.Series(w / w.sum(), index=asset_cols), {
                "success": True,
                "method": "scipy_slsqp",
                "objective": float(res.fun),
                "message": res.message,
            }

    # Fallback random search.
    rng = np.random.default_rng(config.seed)
    best_w = x0
    best_val = obj(x0)

    for _ in range(40_000):
        w = rng.dirichlet(np.ones(n))
        w = np.clip(w, min_weight, max_weight)
        w = w / w.sum()
        val = obj(w)
        if val < best_val:
            best_w = w
            best_val = val

    return pd.Series(best_w, index=asset_cols), {
        "success": False,
        "method": "fallback_random_search",
        "objective": float(best_val),
        "message": "scipy unavailable or failed",
    }


def inverse_vol_weights(cov_daily):
    vol = np.sqrt(np.diag(cov_daily.to_numpy()))
    inv = 1.0 / np.maximum(vol, 1e-12)
    return pd.Series(inv / inv.sum(), index=cov_daily.index)


def gmv_weights(cov_daily, min_weight=0.0, max_weight=0.35):
    n = cov_daily.shape[0]
    cov = cov_daily.to_numpy()
    x0 = np.ones(n) / n

    def obj(w):
        return w.T @ cov @ w

    if SCIPY_AVAILABLE:
        res = minimize(
            obj,
            x0=x0,
            method="SLSQP",
            bounds=[(min_weight, max_weight)] * n,
            constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}],
            options={"maxiter": 1000, "ftol": 1e-12}
        )

        if res.success:
            return pd.Series(res.x / res.x.sum(), index=cov_daily.index)

    inv_cov = np.linalg.pinv(cov)
    raw = inv_cov @ np.ones(n)
    raw = np.clip(raw, min_weight, max_weight)
    raw = raw / raw.sum()
    return pd.Series(raw, index=cov_daily.index)

## 14. Optimised portfolios

We compare:

1. market portfolio;
2. equal weight;
3. inverse volatility;
4. global minimum variance;
5. sample-mean MVO;
6. equilibrium-return MVO;
7. Black-Litterman MVO.

In [ ]:
sample_mu_daily = train_returns.mean() - rf_daily

w_market = market_weights.reindex(asset_cols)
w_equal = pd.Series(1.0 / len(asset_cols), index=asset_cols)
w_inv_vol = inverse_vol_weights(cov_train)
w_gmv = gmv_weights(cov_train, config.min_weight, config.max_weight)

w_sample_mvo, sample_mvo_info = optimise_mean_variance(
    sample_mu_daily.reindex(asset_cols),
    cov_train,
    lambda_used,
    config.min_weight,
    config.max_weight
)

w_equilibrium_mvo, eq_mvo_info = optimise_mean_variance(
    pi_daily.reindex(asset_cols),
    cov_train,
    lambda_used,
    config.min_weight,
    config.max_weight
)

w_bl_mvo, bl_mvo_info = optimise_mean_variance(
    mu_bl_daily.reindex(asset_cols),
    cov_bl_daily,
    lambda_used,
    config.min_weight,
    config.max_weight
)

weights_static = pd.DataFrame({
    "market_portfolio": w_market,
    "equal_weight": w_equal,
    "inverse_vol": w_inv_vol,
    "global_min_variance": w_gmv,
    "sample_mean_mvo": w_sample_mvo,
    "equilibrium_mvo": w_equilibrium_mvo,
    "black_litterman_mvo": w_bl_mvo,
})

weights_static

In [ ]:
plt.figure(figsize=(12, 7))
bottom = np.zeros(weights_static.shape[1])
x = np.arange(weights_static.shape[1])

for asset in weights_static.index:
    plt.bar(x, weights_static.loc[asset], bottom=bottom, label=asset)
    bottom += weights_static.loc[asset].values

plt.xticks(x, weights_static.columns, rotation=45, ha="right")
plt.title("Portfolio Weights")
plt.ylabel("Weight")
plt.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

## 15. Active weights versus market

Black-Litterman is often interpreted as controlled active tilts around the market portfolio.

Active weight:

$$
w^{active}=w-w_m
$$

Large active weights should be explained by views.

In [ ]:
active_weights = weights_static.subtract(w_market, axis=0)

active_weights[["sample_mean_mvo", "equilibrium_mvo", "black_litterman_mvo"]]

In [ ]:
plt.figure(figsize=(10, 6))
active = active_weights["black_litterman_mvo"].sort_values()
plt.barh(active.index, active.values)
plt.axvline(0, linestyle="--")
plt.title("Black-Litterman Active Weights vs Market")
plt.xlabel("Active weight")
plt.ylabel("Asset")
plt.show()

## 16. View satisfaction

Check posterior returns against views:

$$
P\mu_{BL}
$$

Compare to:

$$
Q
$$

The posterior does not necessarily equal the views exactly unless view uncertainty is extremely low.

In [ ]:
def view_satisfaction(P, mu_daily, Q_daily, label):
    implied = P.to_numpy() @ mu_daily.reindex(P.columns).to_numpy()

    return pd.DataFrame({
        "view_name": P.index,
        "model": label,
        "view_target_Q_ann": Q_daily.reindex(P.index).values * config.annualisation,
        "implied_view_return_ann": implied * config.annualisation,
        "difference_ann": implied * config.annualisation - Q_daily.reindex(P.index).values * config.annualisation,
    })


view_sat_pi = view_satisfaction(P, pi_daily, Q_daily, "equilibrium_pi")
view_sat_bl = view_satisfaction(P, mu_bl_daily, Q_daily, "black_litterman")

view_satisfaction_table = pd.concat([view_sat_pi, view_sat_bl], ignore_index=True)

view_satisfaction_table

## 17. In-sample estimates

Using each model's expected returns and covariance, estimate expected return, risk, and utility.

This is not proof of performance, but it checks the optimiser inputs.

In [ ]:
def estimate_portfolio_inputs(weights_df, mu_daily, cov_daily, label):
    rows = []

    for portfolio in weights_df.columns:
        w = weights_df[portfolio].to_numpy()
        ret_ann = portfolio_return(w, mu_daily.reindex(weights_df.index)) * config.annualisation
        vol_ann = portfolio_volatility(w, cov_daily.reindex(index=weights_df.index, columns=weights_df.index)) * np.sqrt(config.annualisation)
        utility_ann = ret_ann - 0.5 * lambda_used * vol_ann**2

        rows.append({
            "portfolio": portfolio,
            "input_model": label,
            "expected_excess_return_ann": ret_ann,
            "expected_vol_ann": vol_ann,
            "expected_utility_ann": utility_ann,
            "effective_n": float(1.0 / np.sum(w**2)),
            "max_weight": float(np.max(w)),
        })

    return pd.DataFrame(rows)


estimate_pi = estimate_portfolio_inputs(weights_static, pi_daily, cov_train, "equilibrium_inputs")
estimate_bl = estimate_portfolio_inputs(weights_static, mu_bl_daily, cov_bl_daily, "black_litterman_inputs")
estimate_sample = estimate_portfolio_inputs(weights_static, sample_mu_daily, cov_train, "sample_mean_inputs")

estimate_summary = pd.concat([estimate_pi, estimate_bl, estimate_sample], ignore_index=True)

estimate_summary.head()

## 18. Out-of-sample test

Apply static train-estimated weights to test returns.

We compare:

- annualised return;
- annualised volatility;
- Sharpe proxy;
- max drawdown;
- tail risk;
- turnover cost.

In [ ]:
def historical_var_cvar(losses, alpha=0.95):
    losses = np.asarray(losses, dtype=float)
    var = np.quantile(losses, alpha)
    tail = losses[losses >= var]
    cvar = tail.mean() if len(tail) else np.nan
    return float(var), float(cvar)


def max_drawdown_from_returns(r):
    nav = (1 + r).cumprod()
    dd = nav / nav.cummax() - 1.0
    return dd, float(dd.min())


def backtest_static_weights(test_returns, weights_df, cost_bps):
    out = pd.DataFrame(index=test_returns.index)
    rows = []

    for portfolio in weights_df.columns:
        w = weights_df[portfolio]
        gross = test_returns @ w

        cost = pd.Series(0.0, index=test_returns.index)
        cost.iloc[0] = w.abs().sum() * cost_bps / 10000.0

        net = gross - cost
        nav = (1 + net).cumprod()
        dd, mdd = max_drawdown_from_returns(net)
        var95, cvar95 = historical_var_cvar(-net, 0.95)
        var99, cvar99 = historical_var_cvar(-net, 0.99)

        out[f"{portfolio}_return"] = net
        out[f"{portfolio}_nav"] = nav

        rows.append({
            "portfolio": portfolio,
            "annualised_return": float(net.mean() * config.annualisation),
            "annualised_vol": float(net.std(ddof=1) * np.sqrt(config.annualisation)),
            "sharpe_proxy": float((net.mean() * config.annualisation - config.risk_free_rate_ann) / (net.std(ddof=1) * np.sqrt(config.annualisation))) if net.std(ddof=1) > 0 else np.nan,
            "max_drawdown": mdd,
            "worst_day": float(net.min()),
            "VaR95": var95,
            "CVaR95": cvar95,
            "VaR99": var99,
            "CVaR99": cvar99,
            "total_return": float(nav.iloc[-1] - 1.0),
            "initial_cost": float(cost.iloc[0]),
        })

    return out, pd.DataFrame(rows).sort_values("sharpe_proxy", ascending=False)


static_backtest, static_performance = backtest_static_weights(
    test_returns,
    weights_static,
    config.transaction_cost_bps
)

static_performance

In [ ]:
plt.figure(figsize=(12, 6))
for portfolio in weights_static.columns:
    plt.plot(static_backtest.index, static_backtest[f"{portfolio}_nav"], label=portfolio)
plt.title("Static Out-of-Sample Growth")
plt.xlabel("Date")
plt.ylabel("Growth")
plt.legend(ncol=2)
plt.show()

plt.figure(figsize=(10, 6))
plot_df = static_performance.sort_values("CVaR95")
plt.barh(plot_df["portfolio"], plot_df["CVaR95"])
plt.title("Out-of-Sample 95% CVaR")
plt.xlabel("CVaR loss")
plt.ylabel("Portfolio")
plt.show()

## 19. Sensitivity to $\tau$

The parameter $\tau$ controls uncertainty in the prior.

Small $\tau$:

- stronger prior;
- less movement from equilibrium.

Large $\tau$:

- weaker prior;
- views have more influence.

We recompute posterior returns and BL weights for different $\tau$.

In [ ]:
tau_grid = [0.005, 0.01, 0.025, 0.05, 0.10, 0.20, 0.50]

tau_rows = []
tau_weights = []

for tau in tau_grid:
    omega_tau = build_omega(P, cov_train, tau, view_confidence)
    mu_tau, cov_tau, _ = black_litterman_posterior(pi_daily, cov_train, P, Q_daily, omega_tau, tau)

    w_tau, info_tau = optimise_mean_variance(
        mu_tau.reindex(asset_cols),
        cov_tau.reindex(index=asset_cols, columns=asset_cols),
        lambda_used,
        config.min_weight,
        config.max_weight
    )

    active_l1 = float(np.abs(w_tau - w_market).sum())

    metrics = estimate_portfolio_inputs(
        pd.DataFrame({"bl_weight": w_tau}),
        mu_tau,
        cov_tau,
        f"tau_{tau}"
    ).iloc[0]

    tau_rows.append({
        "tau": tau,
        "solver_method": info_tau["method"],
        "active_l1_vs_market": active_l1,
        "expected_excess_return_ann": metrics["expected_excess_return_ann"],
        "expected_vol_ann": metrics["expected_vol_ann"],
        "expected_utility_ann": metrics["expected_utility_ann"],
        "effective_n": metrics["effective_n"],
        "max_weight": metrics["max_weight"],
    })

    record = w_tau.to_dict()
    record["tau"] = tau
    tau_weights.append(record)

tau_sensitivity = pd.DataFrame(tau_rows)
tau_weight_table = pd.DataFrame(tau_weights)

tau_sensitivity

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(tau_sensitivity["tau"], tau_sensitivity["active_l1_vs_market"], marker="o")
plt.xscale("log")
plt.title("Sensitivity to Tau")
plt.xlabel("tau")
plt.ylabel("L1 active weight vs market")
plt.show()

## 20. Sensitivity to view confidence

We scale all view confidence levels from weak to strong.

High confidence should produce larger active tilts.

In [ ]:
confidence_scale_grid = [0.25, 0.50, 0.75, 1.00, 1.25, 1.50]

confidence_rows = []
confidence_weights = []

for scale in confidence_scale_grid:
    conf_scaled = (view_confidence * scale).clip(0.01, 0.99)
    omega_conf = build_omega(P, cov_train, config.tau, conf_scaled)

    mu_conf, cov_conf, _ = black_litterman_posterior(pi_daily, cov_train, P, Q_daily, omega_conf, config.tau)

    w_conf, info_conf = optimise_mean_variance(
        mu_conf.reindex(asset_cols),
        cov_conf.reindex(index=asset_cols, columns=asset_cols),
        lambda_used,
        config.min_weight,
        config.max_weight
    )

    active_l1 = float(np.abs(w_conf - w_market).sum())

    confidence_rows.append({
        "confidence_scale": scale,
        "mean_confidence": float(conf_scaled.mean()),
        "active_l1_vs_market": active_l1,
        "max_weight": float(w_conf.max()),
        "effective_n": float(1.0 / np.sum(w_conf.to_numpy()**2)),
        "solver_method": info_conf["method"],
    })

    record = w_conf.to_dict()
    record["confidence_scale"] = scale
    confidence_weights.append(record)

confidence_sensitivity = pd.DataFrame(confidence_rows)
confidence_weight_table = pd.DataFrame(confidence_weights)

confidence_sensitivity

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(confidence_sensitivity["confidence_scale"], confidence_sensitivity["active_l1_vs_market"], marker="o")
plt.title("Sensitivity to View Confidence")
plt.xlabel("Confidence scale")
plt.ylabel("L1 active weight vs market")
plt.show()

## 21. View attribution to active weights

This is a heuristic attribution:

1. compute equilibrium weights;
2. compute BL weights with all views;
3. compute BL weights with one view removed;
4. difference approximates that view's contribution to active allocation.

This is not exact because views interact nonlinearly through optimisation.

In [ ]:
def bl_weights_for_view_subset(view_names):
    P_sub = P.loc[view_names]
    Q_sub = Q_daily.loc[view_names]
    conf_sub = view_confidence.loc[view_names]
    omega_sub = build_omega(P_sub, cov_train, config.tau, conf_sub)
    mu_sub, cov_sub, _ = black_litterman_posterior(pi_daily, cov_train, P_sub, Q_sub, omega_sub, config.tau)

    w_sub, info_sub = optimise_mean_variance(
        mu_sub.reindex(asset_cols),
        cov_sub.reindex(index=asset_cols, columns=asset_cols),
        lambda_used,
        config.min_weight,
        config.max_weight
    )

    return w_sub


all_view_names = list(P.index)
w_all = w_bl_mvo.copy()

view_attribution_rows = []

for removed_view in all_view_names:
    subset = [v for v in all_view_names if v != removed_view]
    w_without = bl_weights_for_view_subset(subset)
    contribution = w_all - w_without

    for asset, value in contribution.items():
        view_attribution_rows.append({
            "removed_view": removed_view,
            "asset": asset,
            "approx_weight_contribution": value
        })

view_weight_attribution = pd.DataFrame(view_attribution_rows)

view_weight_attribution.head()

In [ ]:
view_to_plot = "absolute_trend_4pct"
sub = view_weight_attribution[view_weight_attribution["removed_view"] == view_to_plot].sort_values("approx_weight_contribution")

plt.figure(figsize=(10, 6))
plt.barh(sub["asset"], sub["approx_weight_contribution"])
plt.axvline(0, linestyle="--")
plt.title(f"Approx Active Weight Contribution: {view_to_plot}")
plt.xlabel("Weight contribution")
plt.ylabel("Asset")
plt.show()

## 22. Rolling Black-Litterman allocation

We run a walk-forward allocation:

1. use trailing window for covariance;
2. compute market-implied returns;
3. apply the same views;
4. solve BL MVO;
5. hold until next rebalance;
6. include turnover costs.

This tests allocation stability and implementation realism.

In [ ]:
def rolling_black_litterman_allocation(returns_df, market_weights, config):
    weight_rows = []
    diag_rows = []

    prev_weights = market_weights.reindex(asset_cols).to_numpy()
    current = config.rebalance_window

    while current < len(returns_df):
        window = returns_df.iloc[current - config.rebalance_window:current]
        cov_window, method, shrinkage = estimate_covariance(window)

        market_ret = window @ market_weights
        market_excess = market_ret - config.risk_free_rate_ann / config.annualisation
        lambda_est = market_excess.mean() / market_excess.var(ddof=1)
        lambda_use = config.risk_aversion_default if not np.isfinite(lambda_est) or lambda_est <= 0 else lambda_est

        pi_window = implied_equilibrium_returns(cov_window, market_weights.reindex(cov_window.index), lambda_use)

        P_window, Q_window, conf_window, _ = build_views(asset_cols, pi_window, config)
        omega_window = build_omega(P_window, cov_window, config.tau, conf_window)

        mu_window, cov_bl_window, _ = black_litterman_posterior(
            pi_window,
            cov_window,
            P_window,
            Q_window,
            omega_window,
            config.tau
        )

        w, info = optimise_mean_variance(
            mu_window.reindex(asset_cols),
            cov_bl_window.reindex(index=asset_cols, columns=asset_cols),
            lambda_use,
            config.min_weight,
            config.max_weight
        )

        hold_end = min(current + config.rebalance_step, len(returns_df))

        for date in returns_df.index[current:hold_end]:
            weight_rows.append(pd.Series(w, name=date))

        diag_rows.append({
            "rebalance_date": returns_df.index[current],
            "cov_method": method,
            "shrinkage": shrinkage,
            "lambda_estimated": lambda_est,
            "lambda_used": lambda_use,
            "solver_success": info["success"],
            "solver_method": info["method"],
            "active_l1_vs_market": float(np.abs(w - market_weights).sum()),
            "turnover_vs_previous": float(np.sum(np.abs(w.to_numpy() - prev_weights))),
            "max_weight": float(w.max()),
            "effective_n": float(1.0 / np.sum(w.to_numpy()**2)),
        })

        prev_weights = w.to_numpy()
        current += config.rebalance_step

    weights = pd.DataFrame(weight_rows)
    weights.index.name = "date"
    weights = weights.reindex(returns_df.index).ffill().fillna(0.0)

    diagnostics = pd.DataFrame(diag_rows)

    return weights, diagnostics


rolling_bl_weights, rolling_bl_diagnostics = rolling_black_litterman_allocation(
    returns,
    market_weights,
    config
)

rolling_bl_diagnostics.head()

## 23. Rolling backtest with costs

Weights at $t-1$ are applied to returns at $t$.

Transaction cost:

$$
cost_t = c\sum_i |w_{i,t}-w_{i,t-1}|
$$

In [ ]:
def backtest_dynamic_weights(returns_df, weights_df, cost_bps):
    investable = weights_df.shift(1).fillna(0.0)
    gross = (investable * returns_df).sum(axis=1)

    turnover = weights_df.diff().abs().sum(axis=1).fillna(weights_df.abs().sum(axis=1))
    cost = turnover * cost_bps / 10000.0

    net = gross - cost
    nav = (1 + net).cumprod()

    return pd.DataFrame({
        "gross_return": gross,
        "transaction_cost": cost,
        "net_return": net,
        "nav": nav,
        "turnover": turnover,
        "gross_exposure": investable.abs().sum(axis=1)
    })


# Rolling BL.
rolling_bl_backtest = backtest_dynamic_weights(returns, rolling_bl_weights, config.transaction_cost_bps)

# Static baselines as dynamic frames.
baseline_weight_frames = {
    "market_portfolio": pd.DataFrame(np.tile(w_market.to_numpy(), (len(returns), 1)), index=returns.index, columns=asset_cols),
    "equal_weight": pd.DataFrame(np.tile(w_equal.to_numpy(), (len(returns), 1)), index=returns.index, columns=asset_cols),
    "inverse_vol_static": pd.DataFrame(np.tile(w_inv_vol.to_numpy(), (len(returns), 1)), index=returns.index, columns=asset_cols),
    "global_min_variance_static": pd.DataFrame(np.tile(w_gmv.to_numpy(), (len(returns), 1)), index=returns.index, columns=asset_cols),
    "static_black_litterman": pd.DataFrame(np.tile(w_bl_mvo.to_numpy(), (len(returns), 1)), index=returns.index, columns=asset_cols),
}

rolling_backtests = {
    "rolling_black_litterman": rolling_bl_backtest
}

for name, wf in baseline_weight_frames.items():
    rolling_backtests[name] = backtest_dynamic_weights(returns, wf, config.transaction_cost_bps)


def summarise_backtests(backtests, start_idx):
    rows = []

    for name, bt in backtests.items():
        sub = bt.iloc[start_idx:].copy()
        r = sub["net_return"]
        nav = (1 + r).cumprod()
        dd = nav / nav.cummax() - 1.0
        var95, cvar95 = historical_var_cvar(-r, 0.95)
        var99, cvar99 = historical_var_cvar(-r, 0.99)

        rows.append({
            "portfolio": name,
            "annualised_return": float(r.mean() * config.annualisation),
            "annualised_vol": float(r.std(ddof=1) * np.sqrt(config.annualisation)),
            "sharpe_proxy": float((r.mean() * config.annualisation - config.risk_free_rate_ann) / (r.std(ddof=1) * np.sqrt(config.annualisation))) if r.std(ddof=1) > 0 else np.nan,
            "max_drawdown": float(dd.min()),
            "worst_day": float(r.min()),
            "VaR95": var95,
            "CVaR95": cvar95,
            "VaR99": var99,
            "CVaR99": cvar99,
            "total_return": float(nav.iloc[-1] - 1.0),
            "mean_turnover": float(sub["turnover"].mean()),
            "annualised_cost_drag": float(sub["transaction_cost"].mean() * config.annualisation),
        })

    return pd.DataFrame(rows).sort_values("sharpe_proxy", ascending=False)


rolling_performance = summarise_backtests(rolling_backtests, config.rebalance_window)
rolling_performance

In [ ]:
plt.figure(figsize=(12, 6))
for name, bt in rolling_backtests.items():
    sub = bt.iloc[config.rebalance_window:]
    nav = (1 + sub["net_return"]).cumprod()
    plt.plot(sub.index, nav, label=name, alpha=0.85)
plt.title("Rolling Black-Litterman Backtest")
plt.xlabel("Date")
plt.ylabel("Growth")
plt.legend(ncol=2)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(rolling_bl_backtest.index, rolling_bl_backtest["turnover"].rolling(21).mean())
plt.title("Rolling Black-Litterman 21-Day Average Turnover")
plt.xlabel("Date")
plt.ylabel("Turnover")
plt.show()

## 24. Rolling Black-Litterman weights

Inspect allocation dynamics.

Large weight jumps indicate view/covariance instability or excessive sensitivity.

In [ ]:
plt.figure(figsize=(12, 6))
for asset in asset_cols:
    plt.plot(rolling_bl_weights.index, rolling_bl_weights[asset], label=asset, alpha=0.75)
plt.title("Rolling Black-Litterman Weights")
plt.xlabel("Date")
plt.ylabel("Weight")
plt.legend(ncol=3)
plt.show()

## 25. Stress-regime behaviour

We compare rolling BL and baselines in calm versus stress regimes.

In [ ]:
regime_series = returns_df.set_index("date")["regime"].reindex(returns.index)
stress_type_series = returns_df.set_index("date")["stress_type"].reindex(returns.index)

stress_rows = []

for name, bt in rolling_backtests.items():
    sub = bt.iloc[config.rebalance_window:].copy()
    sub["regime"] = regime_series.reindex(sub.index)
    sub["stress_type"] = stress_type_series.reindex(sub.index)

    for regime_value, group in sub.groupby("regime"):
        stress_rows.append({
            "portfolio": name,
            "regime": int(regime_value),
            "n_days": int(len(group)),
            "mean_return_ann": float(group["net_return"].mean() * config.annualisation),
            "vol_ann": float(group["net_return"].std(ddof=1) * np.sqrt(config.annualisation)),
            "worst_day": float(group["net_return"].min()),
            "mean_turnover": float(group["turnover"].mean()),
        })

stress_regime_report = pd.DataFrame(stress_rows)
stress_regime_report.head()

In [ ]:
plt.figure(figsize=(12, 6))
for name in ["market_portfolio", "static_black_litterman", "rolling_black_litterman"]:
    sub = stress_regime_report[stress_regime_report["portfolio"] == name]
    plt.plot(sub["regime"], sub["vol_ann"], marker="o", label=name)
plt.title("Realised Volatility by Regime")
plt.xlabel("Regime")
plt.ylabel("Annualised volatility")
plt.xticks([0, 1])
plt.legend()
plt.show()

## 26. Governance flags

A Black-Litterman process should be reviewed if:

1. active weights are too large;
2. one view dominates allocation;
3. turnover is too high;
4. out-of-sample performance is worse than market;
5. stress drawdown is large;
6. views are not satisfied in posterior returns;
7. solver fails.

In [ ]:
market_sharpe = rolling_performance[rolling_performance["portfolio"] == "market_portfolio"]["sharpe_proxy"].iloc[0]
market_cvar = rolling_performance[rolling_performance["portfolio"] == "market_portfolio"]["CVaR95"].iloc[0]

governance_rows = []

for _, row in rolling_performance.iterrows():
    portfolio = row["portfolio"]

    if portfolio == "rolling_black_litterman":
        avg_weights = rolling_bl_weights.iloc[config.rebalance_window:].mean()
        active_l1 = float(np.abs(avg_weights - w_market).sum())
    elif portfolio in baseline_weight_frames:
        avg_weights = baseline_weight_frames[portfolio].iloc[0]
        active_l1 = float(np.abs(avg_weights - w_market).sum())
    else:
        avg_weights = pd.Series(np.nan, index=asset_cols)
        active_l1 = np.nan

    governance_rows.append({
        "portfolio": portfolio,
        "sharpe_proxy": row["sharpe_proxy"],
        "CVaR95": row["CVaR95"],
        "max_drawdown": row["max_drawdown"],
        "mean_turnover": row["mean_turnover"],
        "active_l1_vs_market": active_l1,
        "max_avg_weight": float(avg_weights.max()) if avg_weights.notna().all() else np.nan,
        "flag_sharpe_below_market": bool(row["sharpe_proxy"] < market_sharpe and portfolio != "market_portfolio"),
        "flag_cvar_worse_than_market": bool(row["CVaR95"] > market_cvar and portfolio != "market_portfolio"),
        "flag_active_l1_above_60pct": bool(active_l1 > 0.60) if np.isfinite(active_l1) else False,
        "flag_turnover_above_20pct_daily_avg": bool(row["mean_turnover"] > 0.20),
        "flag_drawdown_below_minus_20pct": bool(row["max_drawdown"] < -0.20),
    })

governance_flags = pd.DataFrame(governance_rows)
governance_flags["review_required"] = governance_flags[
    [
        "flag_sharpe_below_market",
        "flag_cvar_worse_than_market",
        "flag_active_l1_above_60pct",
        "flag_turnover_above_20pct_daily_avg",
        "flag_drawdown_below_minus_20pct",
    ]
].any(axis=1)

governance_flags

## 27. Audit checks

Numerical checks:

1. weights sum to one;
2. weights respect bounds;
3. posterior covariance is positive semi-definite;
4. $\Omega$ is positive diagonal;
5. posterior returns are finite;
6. rolling backtest returns are finite.

In [ ]:
audit_rows = []

for portfolio in weights_static.columns:
    w = weights_static[portfolio]
    audit_rows.append({
        "check": f"{portfolio}_weights_sum_to_one",
        "value": float(abs(w.sum() - 1.0)),
        "passed": bool(abs(w.sum() - 1.0) < 1e-8)
    })
    audit_rows.append({
        "check": f"{portfolio}_weights_within_bounds",
        "value": float(max(w.max() - config.max_weight, config.min_weight - w.min())),
        "passed": bool((w.max() <= config.max_weight + 1e-8) and (w.min() >= config.min_weight - 1e-8))
    })

eigvals_bl = np.linalg.eigvalsh(cov_bl_daily.to_numpy())
audit_rows.append({
    "check": "posterior_covariance_psd_tolerance",
    "value": float(eigvals_bl.min()),
    "passed": bool(eigvals_bl.min() > -1e-10)
})

omega_diag_min = float(np.diag(omega.to_numpy()).min())
audit_rows.append({
    "check": "omega_positive_diagonal",
    "value": omega_diag_min,
    "passed": bool(omega_diag_min > 0)
})

posterior_returns_finite = bool(np.isfinite(mu_bl_daily.to_numpy()).all())
audit_rows.append({
    "check": "posterior_returns_finite",
    "value": posterior_returns_finite,
    "passed": posterior_returns_finite
})

finite_backtests = all(np.isfinite(bt["net_return"]).all() for bt in rolling_backtests.values())
audit_rows.append({
    "check": "rolling_backtest_returns_finite",
    "value": bool(finite_backtests),
    "passed": bool(finite_backtests)
})

audit_report = pd.DataFrame(audit_rows)
audit_report

## 28. Practical checklist for Black-Litterman

Before trusting a Black-Litterman allocation:

1. **Check market portfolio**  
   Are market-cap weights sensible?

2. **Check risk aversion**  
   Is $\lambda$ estimated or chosen deliberately?

3. **Check covariance**  
   Is the covariance stable and shrinkage-controlled?

4. **Check equilibrium returns**  
   Do implied returns make economic sense?

5. **Check views**  
   Are views explicit, measurable, and documented?

6. **Check confidence**  
   Does $\Omega$ reflect realistic uncertainty?

7. **Check posterior returns**  
   Are posterior returns plausible?

8. **Check active weights**  
   Are tilts explainable by views?

9. **Check sensitivity**  
   Do $\tau$ and confidence changes destabilise the allocation?

10. **Check out of sample**  
   Does BL improve risk-adjusted results after costs?

## 29. Saving outputs

The notebook saves:

1. synthetic returns;
2. factor returns;
3. asset metadata;
4. train covariance;
5. market weights;
6. risk aversion report;
7. implied equilibrium returns;
8. views $P,Q,\Omega$;
9. Black-Litterman posterior returns and covariance;
10. static weights;
11. active weights;
12. view satisfaction table;
13. estimate summaries;
14. static backtest and performance;
15. tau sensitivity;
16. confidence sensitivity;
17. view attribution;
18. rolling BL weights and diagnostics;
19. rolling backtests and performance;
20. stress-regime report;
21. governance flags;
22. audit report;
23. manifest.

In [ ]:
output_dir = Path("data/processed/black_litterman_allocation_v1")
output_dir.mkdir(parents=True, exist_ok=True)

returns_path = output_dir / "synthetic_returns.csv"
factor_returns_path = output_dir / "factor_returns.csv"
metadata_path = output_dir / "asset_metadata.csv"
cov_train_path = output_dir / "train_covariance.csv"
market_weights_path = output_dir / "market_weights.csv"
risk_aversion_path = output_dir / "risk_aversion_report.csv"
pi_table_path = output_dir / "implied_equilibrium_returns.csv"
views_path = output_dir / "view_table.csv"
P_path = output_dir / "view_matrix_P.csv"
Q_path = output_dir / "view_vector_Q.csv"
omega_path = output_dir / "view_uncertainty_omega.csv"
bl_returns_path = output_dir / "black_litterman_returns.csv"
bl_cov_path = output_dir / "black_litterman_covariance.csv"
weights_static_path = output_dir / "static_weights.csv"
active_weights_path = output_dir / "active_weights.csv"
view_satisfaction_path = output_dir / "view_satisfaction.csv"
estimate_summary_path = output_dir / "estimate_summary.csv"
static_backtest_path = output_dir / "static_backtest.csv"
static_performance_path = output_dir / "static_performance.csv"
tau_sensitivity_path = output_dir / "tau_sensitivity.csv"
tau_weights_path = output_dir / "tau_weight_table.csv"
confidence_sensitivity_path = output_dir / "confidence_sensitivity.csv"
confidence_weights_path = output_dir / "confidence_weight_table.csv"
view_attribution_path = output_dir / "view_weight_attribution.csv"
rolling_weights_path = output_dir / "rolling_black_litterman_weights.csv"
rolling_diagnostics_path = output_dir / "rolling_black_litterman_diagnostics.csv"
rolling_performance_path = output_dir / "rolling_performance.csv"
stress_regime_path = output_dir / "stress_regime_report.csv"
governance_flags_path = output_dir / "governance_flags.csv"
audit_report_path = output_dir / "audit_report.csv"
manifest_path = output_dir / "manifest.json"

returns_df.to_csv(returns_path, index=False)
factor_returns.to_csv(factor_returns_path, index=False)
asset_metadata.to_csv(metadata_path, index=False)
cov_train.to_csv(cov_train_path)
market_weights.to_frame("market_weight").to_csv(market_weights_path)
risk_aversion_report.to_frame("value").to_csv(risk_aversion_path)
pi_table.to_csv(pi_table_path, index=False)
view_table.to_csv(views_path, index=False)
P.to_csv(P_path)
Q_daily.to_frame("Q_daily").to_csv(Q_path)
omega.to_csv(omega_path)
bl_return_table.to_csv(bl_returns_path, index=False)
cov_bl_daily.to_csv(bl_cov_path)
weights_static.to_csv(weights_static_path)
active_weights.to_csv(active_weights_path)
view_satisfaction_table.to_csv(view_satisfaction_path, index=False)
estimate_summary.to_csv(estimate_summary_path, index=False)
static_backtest.to_csv(static_backtest_path)
static_performance.to_csv(static_performance_path, index=False)
tau_sensitivity.to_csv(tau_sensitivity_path, index=False)
tau_weight_table.to_csv(tau_weights_path, index=False)
confidence_sensitivity.to_csv(confidence_sensitivity_path, index=False)
confidence_weight_table.to_csv(confidence_weights_path, index=False)
view_weight_attribution.to_csv(view_attribution_path, index=False)
rolling_bl_weights.to_csv(rolling_weights_path)
rolling_bl_diagnostics.to_csv(rolling_diagnostics_path, index=False)
rolling_performance.to_csv(rolling_performance_path, index=False)
stress_regime_report.to_csv(stress_regime_path, index=False)
governance_flags.to_csv(governance_flags_path, index=False)
audit_report.to_csv(audit_report_path, index=False)

rolling_bt_paths = {}
for name, bt in rolling_backtests.items():
    path = output_dir / f"rolling_backtest_{name}.csv"
    bt.to_csv(path)
    rolling_bt_paths[name] = str(path)

manifest = {
    "dataset_name": "black_litterman_allocation_outputs",
    "schema_version": "black_litterman_allocation_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "asset_cols": asset_cols,
    "split_metadata": split_metadata,
    "covariance_method": cov_method,
    "covariance_shrinkage": cov_shrinkage,
    "lambda_used": float(lambda_used),
    "tau": config.tau,
    "views": view_table.to_dict(orient="records"),
    "sample_mvo_info": sample_mvo_info,
    "equilibrium_mvo_info": eq_mvo_info,
    "black_litterman_mvo_info": bl_mvo_info,
    "rolling_performance": rolling_performance.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "rolling_backtest_files": rolling_bt_paths,
    "notes": [
        "Black-Litterman equilibrium returns are calculated as pi = lambda Sigma w_market.",
        "Investor views are encoded as P mu = Q with view uncertainty Omega.",
        "Omega is built from P tau Sigma P' scaled by view confidence.",
        "Posterior returns and covariance are used in constrained mean-variance optimisation.",
        "Sensitivity to tau and view confidence is reported.",
        "Rolling Black-Litterman allocation includes transaction costs and weights are shifted by one day before applying returns.",
        "This notebook is synthetic and educational; real BL allocation requires carefully governed views and market-cap inputs."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

returns_path, weights_static_path, rolling_performance_path, governance_flags_path, manifest_path

## 30. Limitations

### 30.1 Synthetic data

The notebook uses synthetic returns and synthetic market-cap proxies.

Real allocation requires robust total-return data and realistic investable market weights.

### 30.2 Views are subjective

Black-Litterman does not eliminate subjectivity.

It makes views explicit and uncertainty-aware.

### 30.3 Risk aversion is uncertain

$\lambda$ estimated from sample market returns can be noisy.

### 30.4 $\tau$ is debated

There is no universally correct $\tau$.

Sensitivity analysis is essential.

### 30.5 View uncertainty is hard

$\Omega$ is often more important than the views themselves.

### 30.6 Mean-variance limitations remain

Black-Litterman stabilises $\mu$, but optimisation still depends on covariance, constraints, and model assumptions.

### 30.7 No taxes, liquidity, or full costs

Transaction costs are simplified.

Real portfolios need taxes, liquidity, capacity, turnover limits, and trading constraints.

### 30.8 Views may be correlated

This notebook uses diagonal $\Omega$.

Real views can have correlated errors.

## 31. Research frontier and extensions

Important extensions include:

1. **Idzorek confidence mapping**  
   Convert intuitive confidence levels into $\Omega$.

2. **Factor Black-Litterman**  
   Express views on factors rather than assets.

3. **Bayesian hierarchical views**  
   Model view uncertainty more formally.

4. **Regime-dependent BL**  
   Different priors or views in calm/stress regimes.

5. **Robust BL**  
   Optimise under uncertainty in $\pi$, $P$, $Q$, and $\Omega$.

6. **Transaction-cost-aware BL**  
   Penalise active turnover from current weights.

7. **CVaR Black-Litterman**  
   Use BL returns inside tail-risk optimisation.

8. **Black-Litterman with HRP prior**  
   Use HRP weights as neutral allocation instead of market caps.

9. **Multi-currency BL**  
   Add FX hedging and currency views.

10. **Chinese futures BL**  
   Build equilibrium futures allocation and express views on carry, momentum, term structure, and macro regimes.

## 32. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_13_portfolio_constraints_margin_and_leverage**  
   Add real implementation constraints.

2. **04_14_correlation_breakdown_and_regime_risk**  
   Stress the covariance and views.

3. **05_01_vectorized_backtest_engine**  
   Use BL as a reusable allocation module.

4. **05_06_walk_forward_model_validation**  
   Validate rolling allocation under strict protocols.

5. **07_01_multi_asset_cta_research_pipeline**  
   Blend CTA signals with BL-style priors.

6. **07_03_black_litterman_futures_capstone**  
   Full futures allocation with views on carry, trend, and volatility.

## 33. Summary

This notebook implemented Black-Litterman allocation.

It showed how to:

1. simulate a global multi-asset universe;
2. define market-cap proxy weights;
3. estimate covariance with shrinkage;
4. estimate risk aversion;
5. compute implied equilibrium returns;
6. encode absolute and relative investor views;
7. build $P$, $Q$, and $\Omega$;
8. compute Black-Litterman posterior returns and covariance;
9. optimise constrained portfolios;
10. compare market, equal weight, inverse vol, GMV, sample-MVO, equilibrium-MVO, and BL-MVO;
11. analyse active weights versus market;
12. check view satisfaction;
13. test out of sample;
14. run sensitivity to $\tau$ and confidence;
15. approximate view attribution;
16. run rolling Black-Litterman allocation with costs;
17. create governance flags and audit checks.

The key computational takeaway:

> Black-Litterman combines equilibrium priors and explicit views using Bayesian linear algebra.

The key financial takeaway:

> Black-Litterman does not remove judgement; it forces judgement to be explicit, quantified, and constrained.

## 34. Further reading

- Black and Litterman, "Global Portfolio Optimization."
- He and Litterman on the intuition behind Black-Litterman.
- Idzorek on specifying confidence in Black-Litterman views.
- Meucci, *Risk and Asset Allocation.*
- Grinold and Kahn, *Active Portfolio Management.*
- Literature on Bayesian portfolio construction, equilibrium returns, and robust allocation.